In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname  = "LIG",       # resname of your ligand in the topology
    topology_glob   = "*.pdb",
    trajectory_glob = "*.xtc",
    dt_ns           = 2.0,
)

REPLICA_ROOTS = [
    Path("../run01"),
    Path("../run02"),
]

# Clustering settings
METHOD      = "ward"   # 'ward' or 'kmeans'
N_CLUSTERS  = None     # None = automatic via RMSD_CUTOFF
RMSD_CUTOFF = 2.0      # Å — used when N_CLUSTERS is None

OUTPUT_DIR = Path("./figures")
CLUSTER_PDB_DIR = Path("./cluster_pdbs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLUSTER_PDB_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

## Step 1 — Run clustering for the first replica

In [ ]:
from mdatools.analysis.clustering import PoseClusterer
from mdatools.io.loaders import discover_replicas
from mdatools.universe import load_and_align

clusterer = PoseClusterer(
    cfg,
    method=METHOD,
    n_clusters=N_CLUSTERS,
    rmsd_cutoff=RMSD_CUTOFF,
)

# Run on first replica
rep = discover_replicas(REPLICA_ROOTS, cfg)[0]
u = load_and_align(rep["topology"], rep["trajectory"], cfg)
result = clusterer.run(u, rep["name"])

print(f"Sample: {result.sample_name}")
print(f"Frames: {len(result.labels)}  →  {result.n_clusters} clusters (method={result.method})")
print()
print(result.summary.to_string(index=False))

## Step 2 — Cluster assignment timeline

In [ ]:
from mdatools.plotting.clustering_plots import plot_cluster_timeline

fig = plot_cluster_timeline(
    result,
    save_path=OUTPUT_DIR / f"cluster_timeline_{result.sample_name}.png",
)
fig

## Step 3 — Cluster population

In [ ]:
from mdatools.plotting.clustering_plots import plot_cluster_population

fig = plot_cluster_population(
    result,
    save_path=OUTPUT_DIR / f"cluster_population_{result.sample_name}.png",
)
fig

## Step 4 — Pairwise RMSD matrix

In [ ]:
from mdatools.plotting.clustering_plots import plot_rmsd_matrix

fig = plot_rmsd_matrix(
    result,
    save_path=OUTPUT_DIR / f"rmsd_matrix_{result.sample_name}.png",
)
fig

## Step 5 — Ward dendrogram

In [ ]:
from mdatools.plotting.clustering_plots import plot_dendrogram

fig = plot_dendrogram(
    result,
    save_path=OUTPUT_DIR / f"dendrogram_{result.sample_name}.png",
)
fig

## Step 6 — Extract representative PDBs

In [ ]:
paths = clusterer.extract_representatives(
    u,
    result,
    output_dir=CLUSTER_PDB_DIR / result.sample_name,
)

print(f"Written {len(paths)} representative PDBs:")
for p in paths:
    print(f"  {p}")